# B06：预处理与泄漏诊断（辅修学生练习）

2实验学时。错误示范仅作诊断；修复后继续原项目的一个候选，不读保留测试。聚类/PCA已在理论课堂核验，本Notebook不增加聚类任务。

## 1. 运行前判断

先全开发集填补/标准化再三折验证，会让哪些验证信息进入fit？修复后分数是否一定下降？

**我的判断与依据：**（先填写）

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate

candidates = []
for parent in (Path.cwd(), *Path.cwd().parents):
    candidates.extend(
        [
            parent / "data/preprocessing-classification.csv",
            parent / "units/unsupervised/data/preprocessing-classification.csv",
        ]
    )
data_path = next((p for p in candidates if p.is_file()), None)
if data_path is None:
    raise FileNotFoundError("请保留学生包目录关系")
raw = pd.read_csv(data_path)
dev = raw[raw.split == "train"].reset_index(drop=True)
features = ["planned_load_ratio", "ambient_index"]
X = dev[features]
y = dev.over_limit.to_numpy()
folds = list(StratifiedKFold(n_splits=3, shuffle=True, random_state=42).split(X, y))
assert len(dev) == 120
display(dev.head())

## 2. 故意错误的示范（已提供，不复制到项目）

以下先fit全部120行，再做三折。每个验证折40行的特征已进入统计；结果标为leaked_invalid，只作诊断，不能作为有效候选成绩。

In [ ]:
wrong_preprocess = Pipeline(
    [("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]
)
leaked_X = wrong_preprocess.fit_transform(X)  # 故意错误：验证行参加了fit
leaked = cross_validate(
    KNeighborsClassifier(
        n_neighbors=5,
        weights="uniform",
        metric="euclidean",
        algorithm="brute",
        n_jobs=1,
    ),
    leaked_X,
    y,
    cv=folds,
    scoring="accuracy",
    return_estimator=True,
    error_score="raise",
)

## 3. TODO：修复Pipeline

repaired_model()返回完整Pipeline：imputer为中位数填补；scaler为StandardScaler；model为同参数KNN。此函数不fit，后续cross_validate负责只在每折训练行上fit。

In [ ]:
def repaired_model():
    # TODO：返回完整Pipeline，完成后删除停止提示。
    raise NotImplementedError("请修复预处理范围")
    return Pipeline(
        [
            ("imputer", ...),
            ("scaler", ...),
            (
                "model",
                KNeighborsClassifier(
                    n_neighbors=5,
                    weights="uniform",
                    metric="euclidean",
                    algorithm="brute",
                    n_jobs=1,
                ),
            ),
        ]
    )

In [ ]:
fixed = cross_validate(
    repaired_model(),
    X,
    y,
    cv=folds,
    scoring="accuracy",
    return_estimator=True,
    error_score="raise",
)
rows = []
for name, run in [("leaked_invalid", leaked), ("repaired", fixed)]:
    s = run["test_score"]
    rows.append(
        {
            "workflow": name,
            "fold1": s[0],
            "fold2": s[1],
            "fold3": s[2],
            "cv_mean": s.mean(),
        }
    )
results = pd.DataFrame(rows)
display(results)
ti, vi = folds[0]
fitted = fixed["estimator"][0]
stats = pd.DataFrame(
    {
        "feature": features,
        "wrong_median_all120": wrong_preprocess.named_steps["imputer"].statistics_,
        "correct_median_train80": fitted.named_steps["imputer"].statistics_,
        "wrong_mean_all120": wrong_preprocess.named_steps["scaler"].mean_,
        "correct_mean_train80": fitted.named_steps["scaler"].mean_,
    }
)
display(stats)
for (ti, vi), model in zip(folds, fixed["estimator"]):
    np.testing.assert_allclose(
        model.named_steps["imputer"].statistics_, X.iloc[ti].median().to_numpy()
    )
    train_filled = model.named_steps["imputer"].transform(X.iloc[ti])
    np.testing.assert_allclose(
        model.named_steps["scaler"].mean_, train_filled.mean(axis=0)
    )
print("正确流程统计来自各折80行训练；不能按分数高低判定泄漏。")

## 4. 解释修复

1. 错误fit看到了哪些行？为什么不看标签仍有问题？
2. 引用一项统计与分数，说明修复做了什么；分数是否按你预想变化？
3. PCA若用于分类预测，应放到哪一步、由哪些行fit？

**我的解释：**（填写）

## 项目阶段：保留基线，追加一个候选（20分钟）

按project/README-minor.md操作，将candidate-cells-minor.ipynb追加到本人原项目，先运行B05上文。沿用原划分、主指标与开发数据；不覆盖草稿、不重新抽划分、不读保留测试。使用给定的logistic候选，不改成运行多个候选后择优。保存本实验及原项目到非保护盘并备份。